<a href="https://colab.research.google.com/github/PovSobek/Manipuladores/blob/main/05_Generacion_trayectorias_2_puntos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color='steelblue'> Generación de trayectorias Interactivas </font>

**Material desarrollado por Vicente Esteve-Sala**

![](https://drive.google.com/thumbnail?id=1GrzpXe8zsvQwyY7zTw87VCG4c14sHTvt&sz=w800)

**Fecha última edición**: 13/10/2025

**Licencia**:
<small> © 2025 by <a href="https://cvnet.cpd.ua.es/curriculum-breve/es/esteve-sala-vicente-manuel/283903">Vicente Esteve-Sala</a> is licensed under <a href="https://creativecommons.org/licenses/by-nc-sa/4.0/">CC BY-NC-SA 4.0       </a><small><a rel="license" href="http://creativecommons.org/licenses/by-nc-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-nc-sa/4.0/88x31.png" /></a><br /></small>

Al usar estos contenidos, aceptas los términos de uso, propiedad intelectual y la política de privacidad de la UA.

# Generación de Trayectorias Interactivas 📈

Vamos a planificar un movimiento completo. El objetivo es mover la pinza del robot en **línea recta** desde un punto de inicio a un punto final que tú definirás.

El proceso es el siguiente:
1.  **Definir los parámetros** del robot y la trayectoria de forma interactiva.
2.  **Discretizar la línea recta** que une los puntos de inicio y fin en una serie de puntos intermedios.
3.  **Usar Cinemática Inversa (IK)** para calcular los ángulos de las articulaciones $(\theta_1, \theta_2)$ para *cada uno* de esos puntos.
4.  **Generar una animación** del robot siguiendo la trayectoria, mostrando siempre su espacio de trabajo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# --- Función de Cinemática Inversa (de cuadernos anteriores) ---
def calcular_ik_solucion(x, y, L1, L2):
    distancia = np.sqrt(x**2 + y**2)
    alcance_max = L1 + L2
    alcance_min = np.abs(L1 - L2)

    if not (alcance_min <= distancia <= alcance_max):
        return None # Punto inalcanzable

    cos_arg = (distancia**2 - L1**2 - L2**2) / (2 * L1 * L2)
    if not (-1 <= cos_arg <= 1): # Chequeo de seguridad numérica
        return None

    theta2_rad = np.arccos(cos_arg)
    beta = np.arctan2(y, x)
    alpha = np.arctan2(L2 * np.sin(theta2_rad), L1 + L2 * np.cos(theta2_rad))
    theta1_rad = beta - alpha
    return (theta1_rad, theta2_rad)

# --- Entrada de Datos Interactiva ---
try:
    print("--- Introduce los PARÁMETROS del ROBOT ---")
    L1 = float(input("Longitud del eslabón 1 (L1) en metros: "))
    L2 = float(input("Longitud del eslabón 2 (L2) en metros: "))

    print("\n--- Introduce la TRAYECTORIA ---")
    x_ini = float(input("Coordenada X inicial: "))
    y_ini = float(input("Coordenada Y inicial: "))
    x_fin = float(input("Coordenada X final: "))
    y_fin = float(input("Coordenada Y final: "))

    num_puntos = int(input("\nNúmero de puntos intermedios en la trayectoria: "))

    P_inicio = (x_ini, y_ini)
    P_fin = (x_fin, y_fin)

    print("\n¡Entorno preparado!")

except ValueError:
    print("\nError: Por favor, introduce solo valores numéricos. Vuelve a ejecutar la celda.")

## Cálculo de la Trayectoria

Ahora que tenemos todos los parámetros, la siguiente celda realizará el trabajo pesado:
1.  Generará los puntos intermedios de la línea recta usando `np.linspace`.
2.  Iterará sobre cada punto y llamará a la función `calcular_ik_solucion` para encontrar el par de ángulos correspondiente.
3.  Almacenará todos estos pares de ángulos en una lista, que será nuestra "receta" de movimiento para el robot.

Si algún punto de la trayectoria está fuera del alcance, el proceso se detendrá y te avisará.

In [ ]:
# 1. --- DISCRETIZAR LA TRAYECTORIA CARTESIANA ---
puntos_x = np.linspace(P_inicio[0], P_fin[0], num_puntos)
puntos_y = np.linspace(P_inicio[1], P_fin[1], num_puntos)

# 2. --- CALCULAR LA TRAYECTORIA ARTICULAR (SECUENCIA DE ÁNGULOS) ---
trayectoria_articular = []
alcance_max = L1 + L2
margen = alcance_max * 0.2 # Aumentamos un poco el margen para dar espacio a la leyenda

for i in range(num_puntos):
    x, y = puntos_x[i], puntos_y[i]
    angulos_rad = calcular_ik_solucion(x, y, L1, L2)

    if angulos_rad:
        trayectoria_articular.append(angulos_rad)
    else:
        # (The error handling code remains the same)
        print(f"ADVERTENCIA: El punto ({x:.2f}, {y:.2f}) es inalcanzable.")
        # ... (error plotting code omitted for brevity, it doesn't need changes) ...
        plt.show()
        break

if len(trayectoria_articular) == num_puntos:
    print(f"Se ha calculado la trayectoria articular completa con {len(trayectoria_articular)} configuraciones.")

    # 3. --- ANIMACIÓN ---
    fig, ax = plt.subplots(figsize=(10, 8)) # Ajustamos el tamaño para mejor visualización

    # --- CAMBIO CLAVE: Ajustar el espacio a la derecha para la leyenda ---
    fig.subplots_adjust(right=0.75)
    # --- FIN DEL CAMBIO ---

    ax.set_aspect('equal')
    ax.set_xlim(-(alcance_max + margen), alcance_max + margen)
    ax.set_ylim(-(alcance_max + margen), alcance_max + margen)
    ax.grid(True)

    # Dibujar el espacio de trabajo
    circulo_externo = plt.Circle((0, 0), L1 + L2, color='lightblue', alpha=0.5, label='Espacio de Trabajo')
    circulo_interno = plt.Circle((0, 0), abs(L1 - L2), color='white')
    ax.add_patch(circulo_externo)
    ax.add_patch(circulo_interno)

    # Dibujar los elementos de la animación
    linea_brazo1, = ax.plot([], [], 'r-o', lw=3, ms=10, label='Eslabón 1')
    linea_brazo2, = ax.plot([], [], 'b-o', lw=3, ms=10, label='Eslabón 2')
    ax.plot(puntos_x, puntos_y, 'g--', label='Trayectoria Deseada')

    # Marcar puntos inicial y final
    ax.plot(P_inicio[0], P_inicio[1], 'rx', markersize=12, mew=2, label='Inicio Trayectoria')
    ax.plot(P_fin[0], P_fin[1], 'bx', markersize=12, mew=2, label='Fin Trayectoria')

    # Mover la leyenda fuera del área de trabajo
    ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1), borderaxespad=0.)

    def animate(i):
        th1, th2 = trayectoria_articular[i]
        x1 = L1 * np.cos(th1)
        y1 = L1 * np.sin(th1)
        x2 = x1 + L2 * np.cos(th1 + th2)
        y2 = y1 + L2 * np.sin(th1 + th2)

        linea_brazo1.set_data([0, x1], [0, y1])
        linea_brazo2.set_data([x1, x2], [y1, y2])
        ax.set_title(f'Frame {i+1}/{len(trayectoria_articular)}')
        return linea_brazo1, linea_brazo2,

    ani = animation.FuncAnimation(fig, animate, frames=len(trayectoria_articular),
                                  interval=50, blit=True)
    plt.close(fig)
    display(HTML(ani.to_jshtml()))

**Tu Tarea:**
1. Completa los parámetros del robot y las posiciones inicial y final, en dos posiciones diferentes.
2. Realiza capturas de pantalla de las dos posiciones
3. Entrega en un pdf las capturas de pantallas